# mgdvault tour: from first secret to power-user features

This notebook walks through mgdvault in order of how much you need to know:

1. **Quick start**: one environment variable, then `get` / `set`.
2. **Everyday use**: defaults, JSON token bundles, namespaces, the CLI.
3. **Power user**: explicit vaults, test doubles, atomic `update`, concurrency,
   caching behaviour, alternative backends, and KeePassXC interoperability.

Everything here runs against a **throwaway vault in a temporary directory with
made-up values**, so the saved outputs contain nothing sensitive. In real use the
password comes from your shell profile or from the project's gitignored `.env`,
which mgdvault loads automatically (never overriding variables already exported).
This notebook sets `MGDVAULT_DOTENV=0` so no `.env` on your machine can interfere.

## 1. Quick start

mgdvault needs exactly one thing: `MGDVAULT_PASSWORD`. We also set
`MGDVAULT_PATH` so this demo does not touch your real vault.

In [1]:
import os
import tempfile
from pathlib import Path

demo_dir = Path(tempfile.mkdtemp(prefix="mgdvault-demo-"))

os.environ["MGDVAULT_PASSWORD"] = "demo-password-not-a-real-secret"
os.environ["MGDVAULT_PATH"] = str(demo_dir / "vault.kdbx")
os.environ["MGDVAULT_DOTENV"] = "0"  # ignore any real .env while demoing

import mgdvault

print("mgdvault", mgdvault.__version__)
print(mgdvault.default_vault())

mgdvault 0.1.0
<KeePassVault KeePass database at /var/folders/gc/nhccvkmn7s14lpbh4rjj81ch0000gn/T/mgdvault-demo-x0k42yja/vault.kdbx>


The database does not exist yet; it is created the first time you store something.

In [2]:
mgdvault.set("demo/openai/api_key", "sk-demo-0000000000000000")
mgdvault.get("demo/openai/api_key")

'sk-demo-0000000000000000'

In [3]:
print("file created:", Path(os.environ["MGDVAULT_PATH"]).exists())
print("keys:", mgdvault.list_keys())

file created: True
keys: ['demo/openai/api_key']


That is the whole learning curve for most projects: set the password, call `get`.

## 2. Everyday use

### Missing keys raise by default

A silently missing credential is usually a bug, so `get` raises unless you pass a
`default`.

In [4]:
from mgdvault import SecretNotFoundError

try:
    mgdvault.get("demo/not/configured")
except SecretNotFoundError as exc:
    print("raised:", exc)

print("with default:", mgdvault.get("demo/not/configured", default=None))
print("exists:", mgdvault.exists("demo/openai/api_key"), mgdvault.exists("demo/nope"))

raised: No secret stored at 'demo/not/configured' in KeePass database at /var/folders/gc/nhccvkmn7s14lpbh4rjj81ch0000gn/T/mgdvault-demo-x0k42yja/vault.kdbx.
with default: None
exists: True False


### JSON helpers for token bundles

OAuth libraries hand you dictionaries (for example `Credentials.to_json()` from
google-auth). Store them with `set_json`, read them back with `get_json`.

In [5]:
fake_oauth_bundle = {
    "token": "ya29.demo-access-token",
    "refresh_token": "1//demo-refresh-token",
    "scopes": ["https://www.googleapis.com/auth/gmail.modify"],
    "expiry": "2030-01-01T00:00:00Z",
}
mgdvault.set_json("demo/google/personal", fake_oauth_bundle)

bundle = mgdvault.get_json("demo/google/personal")
print(type(bundle).__name__, "with keys", sorted(bundle))
print("stored as a plain string:", mgdvault.get("demo/google/personal")[:40], "...")

dict with keys ['expiry', 'refresh_token', 'scopes', 'token']
stored as a plain string: {"expiry":"2030-01-01T00:00:00Z","refres ...


### Namespaces

Keys are slash-separated paths. Convention: `<application>/<service>/<name>`.
Many packages can share one vault, and `list_keys(prefix)` shows what an
application owns.

In [6]:
mgdvault.set("demo/ynab/token", "ynab-demo-token")
mgdvault.set("otherapp/postgres/password", "pg-demo-password")

print("everything:   ", mgdvault.list_keys())
print("demo only:    ", mgdvault.list_keys("demo"))
print("demo/google:  ", mgdvault.list_keys("demo/google"))
print("deleted:      ", mgdvault.delete("otherapp/postgres/password"))
print("delete again: ", mgdvault.delete("otherapp/postgres/password"))

everything:    ['demo/google/personal', 'demo/openai/api_key', 'demo/ynab/token', 'otherapp/postgres/password']
demo only:     ['demo/google/personal', 'demo/openai/api_key', 'demo/ynab/token']
demo/google:   ['demo/google/personal']
deleted:       True
delete again:  False


### Invalid keys are rejected early

Segments may contain letters, digits, `_`, `-` and `.`; no spaces, no empty segments.

In [7]:
from mgdvault import InvalidKeyError

for bad in ["has space", "/leading", "double//slash", ""]:
    try:
        mgdvault.set(bad, "x")
    except InvalidKeyError as exc:
        print(f"{bad!r:16} -> {exc}")

'has space'      -> Key segment 'has space' in 'has space' is invalid. Segments may contain letters, digits, '_', '-' and '.', and must start with a letter, digit or '_'.
'/leading'       -> Key '/leading' has an empty segment; keys must not start or end with '/' or contain '//'.
'double//slash'  -> Key 'double//slash' has an empty segment; keys must not start or end with '/' or contain '//'.
''               -> Key must not be empty.


### The command line

The same vault is reachable from the `mgdvault` CLI, which honours the same
environment variables. `set` never takes the value as an argument: interactively
it prompts with hidden input, otherwise it reads stdin or `--from-file`. That
keeps secrets out of shell history and out of agent transcripts.

Below we call the CLI through `subprocess` so the notebook stays reproducible.

In [8]:
import subprocess
import sys


def cli(*args, input=None):
    result = subprocess.run(
        [sys.executable, "-m", "mgdvault.cli", *args],
        input=input, text=True, capture_output=True, env=os.environ,
    )
    out = (result.stdout + result.stderr).rstrip()
    print(f"$ mgdvault {' '.join(args)}\n{out}\n(exit {result.returncode})\n")


cli("info")
cli("set", "demo/maps/api_key", input="AIza-demo-maps-key")
cli("list", "demo")
cli("exists", "demo/maps/api_key")
cli("exists", "demo/missing")

$ mgdvault info
dotenv:    none loaded
backend:   keepass
location:  KeePass database at /var/folders/gc/nhccvkmn7s14lpbh4rjj81ch0000gn/T/mgdvault-demo-x0k42yja/vault.kdbx
file:      exists
secrets:   3
(exit 0)



$ mgdvault set demo/maps/api_key
Stored demo/maps/api_key
(exit 0)



$ mgdvault list demo
demo/google/personal
demo/maps/api_key
demo/openai/api_key
demo/ynab/token
(exit 0)



$ mgdvault exists demo/maps/api_key
yes
(exit 0)



$ mgdvault exists demo/missing
no
(exit 1)



`mgdvault get KEY` prints the value and nothing else, so it composes in shell
pipelines (`export OPENAI_API_KEY="$(mgdvault get demo/openai/api_key)"`). Avoid
running it inside an AI agent session; use `exists` and `list` there instead.

## 3. Power-user features

### Explicit vaults

The module-level functions use one default vault resolved from the environment.
When a program needs several stores, build them directly. Every class shares the
same interface.

In [9]:
from mgdvault import KeePassVault, MemoryVault, Vault

project_vault = KeePassVault(demo_dir / "project.kdbx", password="another-demo-password")
project_vault.set("deploy/token", "deploy-demo-token")

scratch = MemoryVault({"demo/feature_flag": "on"})

for v in (mgdvault.default_vault(), project_vault, scratch):
    assert isinstance(v, Vault)
    print(f"{v.backend_name:8} {len(v):2} secret(s)  {v.describe_location()}")

keepass   4 secret(s)  KeePass database at /var/folders/gc/nhccvkmn7s14lpbh4rjj81ch0000gn/T/mgdvault-demo-x0k42yja/vault.kdbx
keepass   1 secret(s)  KeePass database at /var/folders/gc/nhccvkmn7s14lpbh4rjj81ch0000gn/T/mgdvault-demo-x0k42yja/project.kdbx
memory    1 secret(s)  in-memory vault


### Layering vaults

A common pattern: a per-project vault that falls back to the shared one.

In [10]:
def lookup(key: str) -> str:
    return project_vault.get(key, default=None) or mgdvault.get(key)


print(lookup("deploy/token"))          # from the project vault
print(lookup("demo/openai/api_key"))   # falls back to the shared default vault

deploy-demo-token
sk-demo-0000000000000000


### Swapping the default vault in tests

`set_default_vault` lets a test suite run without any database or OS keyring.

In [11]:
mgdvault.set_default_vault(MemoryVault({"demo/openai/api_key": "test-key"}))
print("during test:", mgdvault.get("demo/openai/api_key"))

mgdvault.reset_default_vault()  # back to environment-based resolution
print("after reset: ", mgdvault.get("demo/openai/api_key"))

during test: test-key


after reset:  sk-demo-0000000000000000


### Atomic read-modify-write with `update`

Pipelines run several processes at once, and OAuth refresh means *read the
bundle, refresh, write it back*. A `get` followed by a `set` is a race: two
processes can both refresh and one overwrites the other. For providers that
rotate refresh tokens (Whoop does), the loser is left holding a dead token.

`update(key, fn)` holds the inter-process write lock across the callback and
re-reads the file first, so the callback always sees the latest value.

In [12]:
import json


def refresh_if_needed(current: str | None) -> str | None:
    bundle = json.loads(current)
    bundle["token"] = "ya29.demo-REFRESHED"
    bundle["refresh_count"] = bundle.get("refresh_count", 0) + 1
    return json.dumps(bundle)


mgdvault.update("demo/google/personal", refresh_if_needed)
print(mgdvault.get_json("demo/google/personal")["token"])

# Returning the current value unchanged performs no write; returning None deletes.
mgdvault.update("demo/google/personal", lambda cur: cur)
print(mgdvault.update("demo/ynab/token", lambda cur: None), mgdvault.exists("demo/ynab/token"))

ya29.demo-REFRESHED
None False


### Concurrency: many writers, one file

Each thread below opens its **own** `KeePassVault` handle on the same file (as
separate processes would) and increments a counter 10 times with `update`. The
file lock serialises them; no increments are lost.

In [13]:
from concurrent.futures import ThreadPoolExecutor

path = os.environ["MGDVAULT_PATH"]
password = os.environ["MGDVAULT_PASSWORD"]
mgdvault.set("demo/counter", "0")


def worker(_):
    handle = KeePassVault(path, password=password)
    for _ in range(10):
        handle.update("demo/counter", lambda cur: str(int(cur) + 1))


with ThreadPoolExecutor(max_workers=5) as pool:
    list(pool.map(worker, range(5)))

print("expected 50, got", mgdvault.get("demo/counter"))

expected 50, got 50


### Caching and change detection

Unlocking a KeePass file runs Argon2 key derivation, which costs a few hundred
milliseconds. mgdvault pays it once per vault instance, then serves reads from
memory. Before every operation it checks the file's modification time; if another
process wrote to it, it re-reads using the cached derived key, which takes about a
millisecond. Repeated reads are effectively free.

In [14]:
import time

fresh = KeePassVault(path, password=password)

t0 = time.perf_counter(); fresh.get("demo/openai/api_key"); first = time.perf_counter() - t0
t0 = time.perf_counter()
for _ in range(500):
    fresh.get("demo/openai/api_key")
hot = time.perf_counter() - t0

print(f"first read (unlock): {first * 1000:6.1f} ms")
print(f"500 cached reads:    {hot * 1000:6.1f} ms total")

# Another handle writes; `fresh` notices on its next read without re-deriving the key.
KeePassVault(path, password=password).set("demo/openai/api_key", "sk-demo-ROTATED")
t0 = time.perf_counter(); value = fresh.get("demo/openai/api_key"); refresh = time.perf_counter() - t0
print(f"read after external write: {refresh * 1000:6.1f} ms ->", value)

first read (unlock):  292.4 ms
500 cached reads:      17.2 ms total


read after external write:    0.6 ms -> sk-demo-ROTATED


### `.env` files are found automatically

`uv run mgdvault ...` and cron jobs do not load a project's `.env` the way an
application calling python-dotenv does, so mgdvault does it itself: the nearest
`.env` at or above the working directory is loaded when the default vault is first
used and at the start of every CLI command. **Already-set variables are never
overridden.** Here we create a fake project with a `.env`, point `MGDVAULT_DOTENV`
at it (normally discovery is automatic from the working directory), and show that
an exported variable still wins.

In [15]:
fake_project = demo_dir / "fake-project"
fake_project.mkdir()
(fake_project / ".env").write_text(
    "MGDVAULT_LOCK_TIMEOUT=5\n"          # not set anywhere else -> adopted
    "MGDVAULT_PASSWORD=ignored-value\n"  # already exported above -> NOT overridden
)

os.environ["MGDVAULT_DOTENV"] = str(fake_project / ".env")
print("loaded:", mgdvault.load_env_file())
print("MGDVAULT_LOCK_TIMEOUT:", os.environ["MGDVAULT_LOCK_TIMEOUT"])
print("password overridden?", os.environ["MGDVAULT_PASSWORD"] == "ignored-value")

mgdvault.reset_default_vault()
print("lock timeout now used by the default vault:", mgdvault.default_vault().lock_timeout)

os.environ["MGDVAULT_DOTENV"] = "0"
del os.environ["MGDVAULT_LOCK_TIMEOUT"]
mgdvault.reset_default_vault()

loaded: /var/folders/gc/nhccvkmn7s14lpbh4rjj81ch0000gn/T/mgdvault-demo-x0k42yja/fake-project/.env
MGDVAULT_LOCK_TIMEOUT: 5
password overridden? False
lock timeout now used by the default vault: 5.0


### Alternative backends

`MGDVAULT_BACKEND` selects the backend for the default vault. `vault_from_env`
builds one from any mapping, which is handy for showing the options without
changing this notebook's environment:

- `keepass` (default): the encrypted file you have been using.
- `keyring`: macOS Keychain, Windows Credential Manager or Linux Secret Service.
  Not executed here because it would write to this machine's real keyring.
- `env`: read-only, for CI systems that inject secrets as variables.
- `memory`: an empty in-process store.

In [16]:
from mgdvault.backends.env import key_to_env_var

env_vault = mgdvault.vault_from_env({
    "MGDVAULT_BACKEND": "env",
    "MGDVAULT_SECRET_DEMO__OPENAI__API_KEY": "sk-from-ci-variables",
})
print(type(env_vault).__name__, "->", env_vault.get("demo/openai/api_key"))
print("env var for a key:", key_to_env_var("demo/google/personal"))

mem = mgdvault.vault_from_env({"MGDVAULT_BACKEND": "memory"})
print(type(mem).__name__, "->", len(mem), "secrets")

try:
    env_vault.set("demo/x", "y")
except mgdvault.ReadOnlyVaultError as exc:
    print("read-only:", exc)

EnvVault -> sk-from-ci-variables
env var for a key: MGDVAULT_SECRET_DEMO__GOOGLE__PERSONAL
MemoryVault -> 0 secrets
read-only: EnvVault is read-only; export MGDVAULT_SECRET_DEMO__X in the environment instead.


### Key files

A KeePass key file can be used alone or together with the password, mapping to
`MGDVAULT_KEYFILE`.

In [17]:
keyfile = demo_dir / "demo.key"
keyfile.write_bytes(os.urandom(32))

kf_vault = KeePassVault(demo_dir / "keyfile.kdbx", password="demo-pw", keyfile=keyfile)
kf_vault.set("demo/thing", "value")

try:
    KeePassVault(demo_dir / "keyfile.kdbx", password="demo-pw").get("demo/thing")
except mgdvault.VaultCredentialsError as exc:
    print("without the key file:", exc)

without the key file: Could not unlock the KeePass database at /var/folders/gc/nhccvkmn7s14lpbh4rjj81ch0000gn/T/mgdvault-demo-x0k42yja/keyfile.kdbx: wrong password or key file (check MGDVAULT_PASSWORD / MGDVAULT_KEYFILE).


### Interoperability with KeePassXC

The vault is a standard KDBX 4 file. Key segments become groups and the last
segment is the entry title, so you can browse or edit it in KeePassXC. Here we
list the structure with pykeepass directly (titles only, no values).

In [18]:
from pykeepass import PyKeePass

kp = PyKeePass(path, password=password)
print("KDBX version", kp.version, "| KDF", kp.kdf_algorithm, "| cipher", kp.encryption_algorithm)
for entry in sorted(kp.entries, key=lambda e: e.path):
    print("  " + "/".join(entry.path))

KDBX version (4, 0) | KDF argon2 | cipher aes256
  demo/counter
  demo/google/personal
  demo/maps/api_key
  demo/openai/api_key


### Error types

Everything derives from `MgdvaultError`, so callers can catch one type when
they do not care about the cause.

In [19]:
import mgdvault as mv

for exc in [mv.VaultConfigurationError, mv.VaultCredentialsError, mv.BackendUnavailableError,
            mv.InvalidKeyError, mv.SecretNotFoundError, mv.ReadOnlyVaultError,
            mv.VaultStorageError, mv.VaultLockTimeoutError]:
    print(f"{exc.__name__:26} -> {[b.__name__ for b in exc.__mro__[1:-1]]}")

missing = {"MGDVAULT_BACKEND": "keepass"}  # no password, no keyfile
try:
    mv.vault_from_env(missing)
except mv.MgdvaultError as exc:
    print("\n" + str(exc))

VaultConfigurationError    -> ['MgdvaultError', 'Exception', 'BaseException']
VaultCredentialsError      -> ['MgdvaultError', 'Exception', 'BaseException']
BackendUnavailableError    -> ['MgdvaultError', 'Exception', 'BaseException']
InvalidKeyError            -> ['MgdvaultError', 'ValueError', 'Exception', 'BaseException']
SecretNotFoundError        -> ['MgdvaultError', 'KeyError', 'LookupError', 'Exception', 'BaseException']
ReadOnlyVaultError         -> ['MgdvaultError', 'Exception', 'BaseException']
VaultStorageError          -> ['MgdvaultError', 'Exception', 'BaseException']
VaultLockTimeoutError      -> ['VaultStorageError', 'MgdvaultError', 'Exception', 'BaseException']

MGDVAULT_PASSWORD is not set. Export it, or put it in a .env file in this project (mgdvault loads the nearest .env automatically), to unlock the KeePass vault, or set MGDVAULT_BACKEND=keyring to use the OS credential manager instead.


## Cleanup

Remove the throwaway vaults. Your real vault, if you have one, was never touched.

In [20]:
import shutil

mgdvault.reset_default_vault()
for var in ("MGDVAULT_PASSWORD", "MGDVAULT_PATH", "MGDVAULT_DOTENV"):
    os.environ.pop(var, None)
shutil.rmtree(demo_dir)
print("removed", demo_dir)

removed /var/folders/gc/nhccvkmn7s14lpbh4rjj81ch0000gn/T/mgdvault-demo-x0k42yja
